# コールグラフ解析と強連結成分（SCC）検出

このノートブックでは、UnifyWeaver の高度なコード解析機能を探求します:

- **コールグラフの構築** - Prolog コードからの依存関係グラフの構築
- **強連結成分（SCC）の検出** - 強連結成分（相互再帰）の検出
- **パターン解析** - 再帰パターンの理解
- **依存関係の視覚化** - 述語間の関係の視覚化

## 学習目標

- UnifyWeaver がコード構造をどのように解析するかを理解する
- コールグラフを構築して検査する
- Tarjan のアルゴリズムを使用して相互再帰を検出する
- コードの依存関係を視覚化する

## セットアップ

UnifyWeaver と解析モジュールをロードします。

In [ ]:
% Load initialization
['../init'].

% Load analysis modules
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## 例 1: シンプルなコールグラフ

シンプルな述語から始めて、そのコールグラフを構築してみましょう。

In [ ]:
% Define ancestor predicate
:- dynamic ancestor/2.
:- dynamic parent/2.

% Parent facts
parent(abraham, isaac).
parent(isaac, jacob).

% Ancestor rules
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### コールグラフの構築

In [ ]:
% Build call graph for ancestor
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 依存関係の解析

In [ ]:
% Get all dependencies of ancestor/2
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% Check if self-recursive
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## 例 2: 相互再帰の検出

偶数・奇数の例を用いて、相互再帰を検出してみましょう。

In [ ]:
% Define mutually recursive predicates
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### 両方の述語に対するコールグラフの構築

In [ ]:
% Build call graph for both predicates
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 強連結成分（SCC）の探索

In [ ]:
% Rebuild the graph because variables do not persist between notebook cells
build_call_graph([is_even/1, is_odd/1], _Graph),
% Find SCCs using Tarjan's algorithm
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### SCC が自明（Trivial）かどうかの確認

In [ ]:
% Rebuild the derived values so this cell also runs independently
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% Check each SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## 例 3: 複雑なコールグラフ

複数の述語を持つより複雑なシステムを解析してみましょう。

In [ ]:
% Define a small program with multiple predicates
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent uses parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: same parent, different children
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: parents are siblings
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### 完全なコールグラフの構築

In [ ]:
% Build call graph for all predicates
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 述語グループの検索

開始述語を含む相互再帰述語グループを検索します。

In [ ]:
% Find the mutually recursive group containing cousin/2
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## 例 4: パターン検出

パターンマッチャーを使用して再帰のタイプを解析します。

In [ ]:
% Define various recursive patterns
:- dynamic count/3.     % Tail recursive
:- dynamic factorial/2. % Linear recursive
:- dynamic fib/2.       % Tree recursive (or linear if detected)

% Tail recursive count
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% Linear recursive factorial
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% Fibonacci (can be detected as linear or tree)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### 末尾再帰の検出

In [ ]:
% Check if count/3 is tail recursive
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### 線形再帰の検出

In [ ]:
% Check if factorial/2 is linear recursive
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### 再帰呼び出しのカウント

In [ ]:
% Count recursive calls in fibonacci
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## DOT 形式による視覚化

コールグラフの Graphviz DOT 表現を生成してみましょう。

In [ ]:
% Helper to generate DOT format
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% Generate DOT for even/odd graph
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### DOT ファイルの保存

In [ ]:
% Rebuild the DOT source because variables do not persist between cells
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## 演習問題: あなた自身のコードを解析してみよう

独自の述語を定義して解析してみましょう！

In [ ]:
% Define your predicates here
% Then build call graphs, find SCCs, and detect patterns

% Example:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## まとめ

このノートブックで学んだこと:

✅ Prolog コードからコールグラフを構築する方法

✅ 相互再帰のための強連結成分（SCC）を検出する方法

✅ パターンマッチャーを使用して再帰タイプを分類する方法

✅ 述語の依存関係を解析する方法

✅ DOT 形式でコールグラフを視覚化する方法

## 発展トピック

より高度な解析のためのトピック:

- **トポロジカルソート**: `topological_order/2` を使用して、依存関係に従って SCC を順序付けする
- **カスタムパターンマッチャー**: 独自のパターン検出述語を作成する
- **アキュムレータパターンの抽出**: 詳細な解析のために `extract_accumulator_pattern/2` を使用する
- **線形再帰の禁止**: `forbid_linear_recursion/1` を使用して異なるコンパイル戦略を強制する

## 参考文献・関連ファイル

- 第10章: Prolog のイントロスペクションと理論
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`